# CoreDNS Attack Reproduction

## 1. Mechanism overview

This section details the mechanism described in the paper and the figures it reports, so they can be directly compared to our own results in section 3.

### 1.1 Two consequences of the same DNS query flood

The paper describes CoreDNS's metrics endpoint as exposed without any access restriction (§3.2):

> "Specifically, CoreDNS exposes the 9153 TCP port on the control plane to provide service metrics information. All the containers in the Kubernetes cluster are able to access this control plane interface without any access restrictions or rate limitations."

This endpoint (TCP, port 9153) is how a malicious container reads CoreDNS's internal counters, but the load that produces the two consequences below comes from a different source: a flood of non-local DNS queries sent to CoreDNS's actual DNS service (UDP, port 53), as carried out in the paper's local-environment experiment (§4.2):

1. Application-level rejection. Each incoming query increments `f.concurrent`; once it exceeds the configured quota, CoreDNS starts refusing further queries and increments `MaxConcurrentRejectCount`, readable through the metrics endpoint.

2. System-level saturation via `nf_conntrack`. The same flood of queries forces CoreDNS to open outbound connections to its upstream resolver for each one, filling up the kernel's connection tracking table and causing packet loss across the whole cluster.

**`launch-attack.sh`** deploys the attack. Before doing so, it restarts the CoreDNS pods (`kubectl delete pod -n kube-system -l k8s-app=kube-dns`). This step is necessary because `coredns_forward_max_concurrent_rejects_total` is a Prometheus counter: it never decreases on its own, even long after an attack has ended, so without a restart every new "before" baseline would inherit rejection counts from whatever ran earlier that day.

**`capture-metrics-loop.sh`** polls the metrics endpoint through the victim pod at a fixed interval, for a fixed duration, and writes each sample to a CSV. Each fetch attempt is explicitly classified as `OK`, `TIMEOUT`, or `ERROR` rather than silently recorded as zero: under heavy load, the control plane can become temporarily unreachable even to `kubectl exec` itself, and a failed fetch is a measurement worth keeping, not a missing one. Only `OK` samples are averaged; `TIMEOUT`/`ERROR` samples are excluded rather than treated as zero-valued readings, since that would understate the actual load rather than reflect the absence of a measurement.

## 3. Results

Measurements from every `before`/`after` run are aggregated and averaged as follows:
### 1.2 The exact rejection mechanism (`ServeDNS`, Fig. 2)

The paper shows the source code of CoreDNS's `forward` plugin to explain precisely when a request gets rejected:

```go
func (f *Forward) ServeDNS(ctx context.Context,
    w dns.ResponseWriter, r *dns.Msg) (int, error) {
  ...
  if f.maxConcurrent > 0 {
    count := atomic.AddInt64(&(f.concurrent), 1)
    defer atomic.AddInt64(&(f.concurrent), -1)
    if count > f.maxConcurrent {
      MaxConcurrentRejectCount.Add(1)
      return dns.RcodeRefused, f.ErrLimitExceeded
    }
  }
  ...
}
```
`f.concurrent` is incremented for every incoming request and decremented once it has been handled. If this counter exceeds `f.maxConcurrent` (the configured quota, `1000` by default in the `Corefile`), the request is immediately refused (`dns.RcodeRefused`) and `MaxConcurrentRejectCount` is incremented. This metric is exposed as `coredns_forward_max_concurrent_rejects_total`, the one we query throughout our captures.

An important point for interpreting the results: this counter does not track how many requests were sent in total, but how many are being processed at the same time, at a given instant. In other words, sending a lot of requests over several minutes is not enough to exceed the quota if they arrive too spread out over time; a large number of them need to genuinely land at the same moment. This distinction turned out to be decisive during the reproduction, as detailed in section 2.

### 1.3 What the paper reports as impact

For the CoreDNS attack in the local environment, the paper reports (§4.2):

> "the victim container suffers 40% packet loss when connecting to the control plane, and the CoreDNS pods consume 90% control plane CPU resources"

These are the two reference figures we compare our own measurements against in section 3.

## 2. Reproduction: manifests and methodology

### 2.1 Attack topology

The attack reuses the namespace and node isolation described in the environment section: attacker pods run in `attacker-ns`, pinned to `k8s-w1` via `nodeName`, while the victim pod runs in `safe-ns`, pinned to `k8s-w2`. Both namespaces enforce the `restricted` Pod Security Standard, so the malicious containers operate under the same constraints assumed by the paper's threat model.

### 2.2 Final manifest

The attack sends a continuous stream of forged, non-local DNS queries (UDP, port 53) directly to the CoreDNS service, matching the mechanism described in §3.2. Each attacker pod spawns several lightweight Python threads that build and send a raw DNS packet in a loop, with a randomized subdomain and source port on every iteration. The full manifest (`dns-flooder.yaml`, `replicas: 3`, `THREADS: 3` per the tuning described in section 2.3) is in the `manifests/` folder.

The attack sends a continuous stream of forged, non-local DNS queries (UDP, port 53) directly to the CoreDNS service, matching the mechanism described in §3.2. Each attacker pod spawns several lightweight Python threads, each opening a raw UDP socket and sending a crafted DNS query packet in a tight loop. The query targets a random 8-character subdomain of `example.com` (e.g. `qzrfmowx.example.com`), a name CoreDNS cannot resolve from its local Kubernetes service records, so every query is treated as non-local and forwarded to the upstream resolver, which is exactly the code path that increments `f.concurrent` in `ServeDNS` (Fig. 2). Each thread rewrites only the random subdomain and the DNS transaction ID between sends, reusing the rest of the packet and opening a new source port each time, which keeps the per-query work cheap enough for many threads to sustain a high query rate concurrently. The full manifest (`dns-flooder-udp.yaml`, `replicas: 3`, `THREADS: 3` per the tuning described in section 2.3) is in the `manifests/` folder.


### 2.3 Why this design and not an earlier one

This manifest is the result of several iterations, not the first approach tried. Two earlier designs looked reasonable on paper but turned out to under-deliver the intended load:

**Shell loops with a process-count check.** An earlier version used `nc`/`nslookup` in a `while` loop, gated by counting active processes (`jobs -p | wc -l`, then `pgrep`) to keep a target number of concurrent requests in flight. In practice, `jobs -p` proved unreliable in a non-interactive `ash` shell, and even once switched to `pgrep`, the loop spent more time forking and counting processes than actually sending requests, capping the real concurrency far below what the loop appeared to request.

**Rebuilding the DNS packet on every iteration.** A second version generated a fresh random subdomain and re-encoded the full DNS packet (string formatting, label encoding, `struct.pack`) on every loop iteration, across many threads. Under Python's GIL, this pure-Python work became the bottleneck: more threads meant more contention for the interpreter lock, not more throughput, and CPU usage climbed to over 90% system time without moving the needle on `nf_conntrack` or `rejects_total`.

The final design fixes both issues at once: threads instead of processes remove the fork/exec cost entirely, and the DNS packet is built once and only patched in place (a handful of bytes rewritten per iteration) rather than rebuilt from scratch, keeping the per-iteration Python work minimal enough that adding threads translates into real additional throughput instead of lock contention.

**Tuning the load to stay measurable.** With `replicas: 7` and `THREADS: 8` (7 pods matching the paper's setup, each spawning 8 threads), the flood saturated `nf_conntrack` and drove the control plane unreachable so quickly and so completely that most `capture-metrics-loop.sh` samples came back as `TIMEOUT` or `ERROR` rather than `OK`, leaving too few usable data points to compute a meaningful average. We reduced the load to `replicas: 3` and `THREADS: 3`, which still reliably triggers both mechanisms described in the paper (a non-zero `MaxConcurrentRejectCount` and a saturated `nf_conntrack` table) but keeps enough samples in the `OK` state throughout the capture window to average. We did not push the load back up past this point once it produced stable, usable figures, since the goal was a reproducible measurement, not the maximum achievable disruption.

### 2.4 Measurement scripts

Three scripts drive the measurement, kept deliberately separate from the attack manifest itself:

**`launch-attack.sh`** deploys the attack. Before doing so, it restarts the CoreDNS pods (`kubectl delete pod -n kube-system -l k8s-app=kube-dns`). This step is necessary because `coredns_forward_max_concurrent_rejects_total` is a Prometheus counter: it never decreases on its own, even long after an attack has ended, so without a restart every new "before" baseline would inherit rejection counts from whatever ran earlier that day.

**`capture-metrics-loop.sh`** polls the metrics endpoint through the victim pod at a fixed interval, for a fixed duration, and writes each sample to a CSV. Each fetch attempt is explicitly classified as `OK`, `TIMEOUT`, or `ERROR` rather than silently recorded as zero: under heavy load, the control plane can become temporarily unreachable even to `kubectl exec` itself, and a failed fetch is a measurement worth keeping, not a missing one. Only `OK` samples are averaged; `TIMEOUT`/`ERROR` samples are excluded rather than treated as zero-valued readings, since that would understate the actual load rather than reflect the absence of a measurement.

**`capture-cpu-loop-coredns.sh`** measures CPU usage of the CoreDNS process specifically, run on k8s-cp, matching the paper's own attribution for this attack ("the CoreDNS pods consume 90% control plane CPU resources", Section 4.2), rather than whole-node CPU. It reads `/proc/<pid>/stat` twice, once at the start of each interval and once at the end, and computes the CPU-tick delta between the two.
## 3. Results

Measurements from every `before`/`after` run are aggregated and averaged as follows:


In [5]:
from aggregate_averages_coredns import find_csv_files, COLS, TCP_COL, LABELS
import pandas as pd
from IPython.display import display, HTML

def load_combined(label):
    csv_files = find_csv_files(label, base_dir="../results")
    dfs = [pd.read_csv(f) for f in csv_files]
    return pd.concat(dfs, ignore_index=True)

before = load_combined("before")
after = load_combined("after")

rows = COLS + [TCP_COL]
summary = pd.DataFrame({
    "Before": before[rows].mean().round(1),
    "After": after[rows].mean().round(1),
}).rename(index=LABELS)

display(HTML("<h2>Metrics comparison before / after attack</h2>"))
display(summary.style.set_properties(**{"font-size": "16pt"}).set_table_styles(
    [{"selector": "th", "props": [("font-size", "16pt")]}]
))

,Before,After
MaxConcurrentRejectCount,0.000000,4394938.400000
nf_conntrack_count,406.500000,131068.600000
CPU user (%),2.700000,21.600000
CPU system (%),3.500000,15.900000
DNS requests (total),1.000000,4982946.200000
DNS responses SERVFAIL,0.000000,65309.100000
CPU CoreDNS process (%),0.000000,143.000000
TCP packet loss to control plane (%),0.000000,25.400000



The table above summarizes the average of every `before`/`after` run collected via `capture-metrics-loop.sh`, including the TCP packet-loss probe added to the same pipeline. The figures below compare it directly against what the paper reports for the same attack in its local environment.

| Metric | Paper (§4.2) | Our reproduction |
|---|---|---|
| CPU consumed by CoreDNS | up to 90% | 143.0% (from 0% at rest) |
| Packet loss on the victim container | up to 40% | 25.4% on average (TCP probes to the control plane, `tcp_probe_status == OK` samples only) |
| `MaxConcurrentRejectCount` | reported only as "becomes a non-zero number", no figure given | average 4,394,938 rejections per sample, against a 0 baseline |
| `nf_conntrack_count` | described qualitatively as filling up, no figure given | 131,069 on average (99.997% of the 131,072 ceiling), against a 406.5 baseline, roughly a 322-fold increase |

Two things stand out from this comparison.

**Packet loss on the victim container is lower in our reproduction than the paper's figure.** Where the paper reports up to 40% loss, our TCP probes against the control plane averaged 25.4% loss during the attack window, against a 0% baseline. This average is computed only over samples where the probe itself succeeded (`tcp_probe_status == OK`), so it is not inflated by missing measurements, but it does span the whole capture window, including the early seconds of each run before `nf_conntrack` reaches its ceiling. Since the table also shows `nf_conntrack_count` sitting at 99.997% of its limit on average, the loss rate at the moment of full saturation is likely higher than this window-wide average suggests, closer to the paper's figure than the aggregate number implies on its own.
This gap is also consistent with a deliberate methodology choice described in section 2.3: to keep enough samples in the `OK` state to average, the final attack configuration (`replicas: 3`, `THREADS: 3`) is less aggressive than the paper's 7-container setup would produce at full intensity. A heavier load would likely push both the packet-loss and CPU figures closer to the paper's, at the cost of losing most measurable samples to `TIMEOUT`/`ERROR`.

**CPU consumption was first measured against the whole control plane node, then re-measured against the CoreDNS process specifically.** The node-wide reading (`cpu_us` + `cpu_sy` via `top`, section 2.4) averaged 37.5% during the attack, well below the paper's 90%. Re-reading the paper's own wording, "the CoreDNS pods consume 90% control plane CPU resources" (§4.2), showed the figure is attributed to the CoreDNS pod specifically, not the node as a whole, so the node-wide reading was not the right point of comparison. `cpu_coredns` (read from `/proc/<pid>/stat`, section 2.4) was added to measure the CoreDNS process directly instead. This second measurement averages 143.0%, above the paper's 90%. The paper's own Fig. 5 shows all seven of its attacks staying under 100%, which suggests its CPU figures are normalized across all cores (capped at 100% by construction), unlike our per-process reading, which can exceed 100% when a process uses more than one core at once, as CoreDNS does here. The two figures likely sit on different scales, which limits how directly they can be compared.

**`MaxConcurrentRejectCount` confirms the rejection mechanism.** It moves from 0 at rest to an average in the millions during the attack, and `nf_conntrack_count` sits essentially at its ceiling, both consistent with section 1.2.